In [5]:
# Task 1.1

import csv
import json

def parse_row(row):
    return {
        **row,
        "age": int(row["age"]),
        "fee_inr": float(row["fee_inr"]),
        "prior_no_shows": int(row["prior_no_shows"]),
        "distance_km": float(row["distance_km"]),
    }

accepted = []
rejected = []


with open("appointments.csv", newline="", encoding="utf-8") as f:
  reader = csv.DictReader(f)

  headers = reader.fieldnames
  print("Headers:", headers)

  row_count = 0

  for row in reader:
    row_count += 1

    try:
      parsed = parse_row(row)
      accepted.append(parsed)
    except ValueError as e:
      row["_error"] = str(e)
      rejected.append(row)

print("Row count:", row_count)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988
Rejected: 32


In [15]:
# Task 1.2

import pandas as pd

# Load data
df = pd.read_csv("appointments.csv")

# Initial inspection
print("Shape:")
print(df.shape)

print("\nInfo:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())



Shape:
(1020, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
None

Missing values:
appointment_id      0
patient_id          0
c

In [14]:

# ---------------------------
# Step (a): Drop exact duplicates
# ---------------------------
df = df.drop_duplicates()
print("\nAfter dropping duplicate rows:", df.shape)


After dropping duplicate rows: (990, 14)


In [23]:
# ---------------------------
# Step (b): Clean booking_channel
# ---------------------------
df["booking_channel"] = (
    df["booking_channel"]
      .str.strip()
      .str.lower()
      .replace({
          "app": "App",
          "phone": "Phone",
          "walk-in": "Walk-In",
          "walk in": "Walk-In",
          "walkin": "Walk-In"
      })
)

print("\nAfter cleaning booking_channel:", df.shape)


After cleaning booking_channel: (1020, 14)


In [24]:
# ---------------------------
# Step (c): Convert appt_date
# ---------------------------
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

print("\nAfter converting appt_date:", df.shape)


After converting appt_date: (1020, 14)


In [25]:
# ---------------------------
# Step (d): Convert booked_on
# ---------------------------
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce"
)

print("\nAfter converting booked_on:", df.shape)


After converting booked_on: (1020, 14)


In [26]:
# ---------------------------
# Step (e): Drop rows where
# appt_date is missing after conversion
# OR status is missing
# ---------------------------
df = df.dropna(
    subset=["appt_date", "status"]
)

print("\nAfter dropping missing appt_date/status:", df.shape)

print("\nFinal shape:", df.shape)


After dropping missing appt_date/status: (1010, 14)

Final shape: (1010, 14)


In [27]:
# Task 1.3

import numpy as np
import pandas as pd

# ---------------------------
# (a) Invalid age -> missing
# ---------------------------
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [28]:
# ---------------------------
# (b) Fill gaps
# ---------------------------

# age -> median, then convert to int
df["age"] = (
    df["age"]
    .fillna(df["age"].median())
    .round()
    .astype(int)
)

In [29]:
# distance_km -> median
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

In [30]:
# ---------------------------
# (c) Outlier capping (IQR)
# ---------------------------

def cap_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    before = df[col].copy()

    df[col] = np.clip(df[col], lower, upper)

    capped_count = (before != df[col]).sum()

    return capped_count

distance_capped = cap_outliers(df, "distance_km")
fee_capped = cap_outliers(df, "fee_inr")

In [31]:
# ---------------------------
# (d) Rounding
# ---------------------------

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"]
    .round()
    .astype(int)
)

In [32]:
# ---------------------------
# Required output
# ---------------------------

print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 0
Fee values capped: 0


In [35]:
# Task 1.4

# ---------------------------
# Feature engineering
# ---------------------------

df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

df["no_show"] = (df["status"] == "No-Show").astype(int)

# ---------------------------
# NumPy matrix
# ---------------------------

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# ---------------------------
# (a) Mean of each column
# ---------------------------

column_means = X.mean(axis=0)

print("Age mean:", column_means[0])
print("Distance mean:", column_means[1])
print("Fee mean:", column_means[2])
print("Lead-days mean:", column_means[3])

# ---------------------------
# (b) Rows with lead_days > 14
# ---------------------------

mask = X[:, 3] > 14

count_gt14 = mask.sum()
share_gt14 = mask.mean()

print("Rows with lead_days > 14:", count_gt14)
print("Share with lead_days > 14:", share_gt14)

# ---------------------------
# (c) Z-scores for distance_km
# ---------------------------

distance = X[:, 1]

z_scores = (distance - distance.mean()) / distance.std()

outlier_count = (np.abs(z_scores) > 2).sum()

print("Rows with |z| > 2:", outlier_count)

Age mean: 48.12178217821782
Distance mean: 9.494059405940595
Fee mean: 636.5346534653465
Lead-days mean: 11.823762376237624
Rows with lead_days > 14: 301
Share with lead_days > 14: 0.298019801980198
Rows with |z| > 2: 54
